<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/banking77-intent-classification/notebooks/pretrained_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/TheFaridKarami/farid-ai-lab.git

Cloning into 'farid-ai-lab'...
remote: Enumerating objects: 242, done.
remote: Counting objects: 100% (242/242), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 242 (delta 96), reused 73 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (242/242), 12.72 MiB | 20.58 MiB/s, done.
Resolving deltas: 100% (96/96), done.


In [2]:
import numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from collections import Counter
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          DataCollatorWithPadding,
                          TrainingArguments,
                          Trainer,
                          set_seed)

SEED = 42
set_seed(SEED)

In [3]:
PROJECT_ROOT = (Path("/content/farid-ai-lab")
 / "projects/nlp/banking77-intent-classification")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

In [4]:
print(PROJECT_ROOT.exists())
print(DATA_DIR.exists())
print(TRAIN_PATH.exists())
print(TEST_PATH.exists())

True
True
True
True


In [5]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Shapes:", train_df.shape, test_df.shape)
print(train_df.dtypes)
print("Nulls:\n", train_df.isnull().sum())
print(train_df.head())

counts = train_df["label"].value_counts()
print("Classes train/test:", train_df["label"].nunique(), test_df["label"].nunique())
print(counts.describe())
print(counts.tail(5))
print("Missing from test:", set(train_df["label"]) - set(test_df["label"]))

print("Duplicate rows:", train_df.duplicated().sum())
print("Duplicate texts train/test:", train_df["text"].duplicated().sum(), test_df["text"].duplicated().sum())
print("Train/test text overlap:", len(set(train_df["text"]) & set(test_df["text"])))
print("Texts with conflicting labels:", (train_df.groupby("text")["label"].nunique() > 1).sum())

Shapes: (10003, 2) (3080, 2)
text     object
label    object
dtype: object
Nulls:
 text     0
label    0
dtype: int64
                                                text         label
0                     I am still waiting on my card?  card_arrival
1  What can I do if my card still hasn't arrived ...  card_arrival
2  I have been waiting over a week. Is the card s...  card_arrival
3  Can I track my card while it is in the process...  card_arrival
4  How do I know if I will get my card, or if it ...  card_arrival
Classes train/test: 77 77
count     77.000000
mean     129.909091
std       32.942207
min       35.000000
25%      112.000000
50%      127.000000
75%      159.000000
max      187.000000
Name: count, dtype: float64
label
lost_or_stolen_card         82
card_swallowed              61
card_acceptance             59
virtual_card_not_working    41
contactless_not_working     35
Name: count, dtype: int64
Missing from test: set()
Duplicate rows: 0
Duplicate texts train/test: 0 0
Trai

In [6]:
train_df = pd.read_csv(TRAIN_PATH).drop_duplicates(subset="text").reset_index(drop=True)
test_df  = pd.read_csv(TEST_PATH)
assert len(train_df) == 10003, len(train_df)

X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    train_df["text"], train_df["label"],
    test_size=0.2, random_state=SEED, stratify=train_df["label"])

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(test_df["label"])

NUM_LABELS = len(le.classes_)
label2id = {str(l): int(i) for i, l in enumerate(le.classes_)}
id2label = {i: l for l, i in label2id.items()}

train_ds = Dataset.from_dict({"text": X_train.tolist(), "labels": y_train.tolist()})
val_ds   = Dataset.from_dict({"text": X_val.tolist(),   "labels": y_val.tolist()})
test_ds  = Dataset.from_dict({"text": test_df["text"].tolist(), "labels": y_test.tolist()})
print(NUM_LABELS, len(train_ds), len(val_ds), len(test_ds))

77 8002 2001 3080


In [7]:
MODEL_NAME = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lengths = np.array([len(ids) for ids in tokenizer(X_train.tolist())["input_ids"]])

print(pd.Series(lengths).describe(percentiles=[.5, .9, .95, .99]))

MAX_LENGTH = 64
print("Coverage at MAX_LENGTH:", f"{(lengths <= MAX_LENGTH).mean().round(4)}%")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

count    8002.000000
mean       16.287178
std         9.322936
min         4.000000
50%        13.000000
90%        28.000000
95%        37.000000
99%        54.000000
max        93.000000
dtype: float64
Coverage at MAX_LENGTH: 0.9969%


In [8]:
def tokenize(batch):
  return tokenizer(batch["text"], max_length = MAX_LENGTH, truncation = True)

train_tok = train_ds.map(tokenize, batched = True, remove_columns = ["text"])
val_tok = val_ds.map(tokenize, batched = True, remove_columns = ["text"])
test_tok = test_ds.map(tokenize, batched = True, remove_columns = ["text"])

collector = DataCollatorWithPadding(tokenizer = tokenizer)

Map:   0%|          | 0/8002 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

In [9]:
def compute_metrics(eval_pred):
   logits, labels = eval_pred
   predictions = np.argmax(logits, axis= -1)
   return {
       "Accuracy_score":accuracy_score(labels, predictions),
       "F1_score":f1_score(labels, predictions, average="macro")
   }

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = NUM_LABELS,
    label2id = label2id,
    id2label= id2label)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
args = TrainingArguments(
    output_dir="/content/outputs/distilbert-banking77",
    num_train_epochs=8,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps = 25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="F1_score",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

In [12]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collector,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result.metrics)

Epoch,Training Loss,Validation Loss,Accuracy Score,F1 Score
1,No log,1.350880,0.785107,0.732716
2,1.902998,0.521726,0.880060,0.868484
3,1.902998,0.379186,0.902049,0.903885
4,0.247679,0.348995,0.912044,0.912984
5,0.247679,0.359325,0.912544,0.915670
6,0.065928,0.365763,0.909545,0.911640
7,0.065928,0.357129,0.912544,0.914503
8,0.021967,0.359304,0.911544,0.913130


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': 207.6574, 'train_samples_per_second': 308.277, 'train_steps_per_second': 9.67, 'total_flos': 772728751122468.0, 'train_loss': 0.5574656599071396, 'epoch': 8.0}


In [13]:
val_metrics = trainer.evaluate(val_tok)
print(val_metrics)

Training Loss,Validation Loss,Epoch,Accuracy Score,F1 Score
0.021967,0.359325,8,0.912544,0.915670


{'eval_loss': 0.3593246638774872, 'eval_Accuracy_score': 0.912543728135932, 'eval_F1_score': 0.9156702066340762}


In [14]:
pred_out  = trainer.predict(val_tok)
val_preds = np.argmax(pred_out.predictions, axis=-1)
val_true = pred_out.label_ids

In [15]:
names = [id2label[i] for i in range(NUM_LABELS)]

report = classification_report(
    val_true,
    val_preds,
    target_names = names,
    labels = list(range(NUM_LABELS)),
    output_dict= True, zero_division=0)
per_class = pd.DataFrame(report).T.iloc[:NUM_LABELS].sort_values("f1-score")

pairs = Counter((id2label[t], id2label[p]) for t, p in zip(val_true, val_preds) if t != p )

for (t, p), n in pairs.most_common(20):
  print(f"{n}  {t}  -> {p}")

6  card_payment_wrong_exchange_rate  -> wrong_exchange_rate_for_cash_withdrawal
5  fiat_currency_support  -> supported_cards_and_currencies
5  direct_debit_payment_not_recognised  -> card_payment_not_recognised
4  why_verify_identity  -> verify_my_identity
4  beneficiary_not_allowed  -> failed_transfer
4  card_delivery_estimate  -> card_arrival
4  pending_top_up  -> top_up_failed
4  declined_transfer  -> failed_transfer
3  Refund_not_showing_up  -> reverted_card_payment?
3  declined_transfer  -> declined_card_payment
3  balance_not_updated_after_bank_transfer  -> transfer_not_received_by_recipient
2  balance_not_updated_after_cheque_or_cash_deposit  -> pending_top_up
2  failed_transfer  -> transfer_not_received_by_recipient
2  why_verify_identity  -> unable_to_verify_identity
2  verify_source_of_funds  -> balance_not_updated_after_cheque_or_cash_deposit
2  pending_card_payment  -> declined_card_payment
2  card_linking  -> lost_or_stolen_card
2  top_up_by_bank_transfer_charge  -> top_up

In [16]:
errors = pd.DataFrame({
    "text":val_ds["text"],
    "True":[id2label[i] for i in val_true],
    "Pred":[id2label[i] for i in val_preds]
})
errors = errors[errors["True"] != errors["Pred"]]



In [17]:
print(f"{len(errors)} errors out of {len(val_ds)}")
print(len(errors))

175 errors out of 2001
175


In [18]:
print(errors.sample(min(20, len(errors)), random_state = SEED).to_string())

                                                                                                                            text                                     True                                     Pred
1682                                                                                        Am I supposed to verify my identity?                      why_verify_identity                       verify_my_identity
1581                                           My credit card transaction was declined for a loan payment.  Can you tell me why?                            top_up_failed                    declined_card_payment
1065                                                                                                  When will my card be here?                   card_delivery_estimate                             card_arrival
1352                                                                                                     Do you support top ups?                          to

In [19]:
test_evaluation = trainer.evaluate(test_tok)
pred_out = trainer.predict(test_tok)
test_preds = np.argmax(pred_out.predictions, axis= -1)
test_true = pred_out.label_ids

Training Loss,Validation Loss,Epoch,Accuracy Score,F1 Score
0.021967,0.311524,8,0.917532,0.917597


# Classification Report

In [20]:
report = classification_report(
    test_true,
    test_preds,
    target_names = [id2label[i] for i in range(NUM_LABELS)],
    output_dict =True,
    zero_division = 0,
    labels= list(range(NUM_LABELS))
)

report_df = pd.DataFrame(report).T.iloc[:NUM_LABELS].sort_values("f1-score")
print(report_df)

                                         precision  recall  f1-score  support
why_verify_identity                       0.875000   0.700  0.777778     40.0
pending_transfer                          0.837838   0.775  0.805195     40.0
declined_transfer                         0.965517   0.700  0.811594     40.0
balance_not_updated_after_bank_transfer   0.909091   0.750  0.821918     40.0
topping_up_by_card                        0.825000   0.825  0.825000     40.0
...                                            ...     ...       ...      ...
passcode_forgotten                        1.000000   1.000  1.000000     40.0
age_limit                                 1.000000   1.000  1.000000     40.0
apple_pay_or_google_pay                   1.000000   1.000  1.000000     40.0
verify_source_of_funds                    1.000000   1.000  1.000000     40.0
verify_top_up                             1.000000   1.000  1.000000     40.0

[77 rows x 4 columns]


In [21]:
pairs = Counter((id2label[t], id2label[p]) for t, p in zip(test_true, test_preds) if t != p)

for (t, p) , n in pairs.most_common(20):
  print(f"{n:>3}  {t} ---> {p}")

  9  why_verify_identity ---> verify_my_identity
  6  pending_transfer ---> transfer_not_received_by_recipient
  5  card_delivery_estimate ---> card_arrival
  4  top_up_by_bank_transfer_charge ---> transfer_fee_charged
  4  compromised_card ---> lost_or_stolen_card
  4  declined_transfer ---> declined_card_payment
  4  declined_transfer ---> failed_transfer
  4  Refund_not_showing_up ---> reverted_card_payment?
  4  balance_not_updated_after_bank_transfer ---> transfer_timing
  3  card_arrival ---> card_delivery_estimate
  3  fiat_currency_support ---> exchange_via_app
  3  pin_blocked ---> change_pin
  3  wrong_amount_of_cash_received ---> declined_cash_withdrawal
  3  top_up_reverted ---> top_up_failed
  3  card_payment_not_recognised ---> compromised_card
  3  why_verify_identity ---> unable_to_verify_identity
  3  virtual_card_not_working ---> card_not_working
  3  balance_not_updated_after_bank_transfer ---> transfer_not_received_by_recipient
  3  balance_not_updated_after_bank_tr

In [23]:
from transformers import pipeline
trainer.save_model("/content/outputs/distilbert-banking77-best")

clf = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer,
               device=0 if torch.cuda.is_available() else -1)
print(clf("My card hasn't arrived yet, when will it get here?", top_k=3))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[{'label': 'card_arrival', 'score': 0.9112858772277832}, {'label': 'card_delivery_estimate', 'score': 0.07778271287679672}, {'label': 'transfer_not_received_by_recipient', 'score': 0.001505454652942717}]


In [26]:
import json
results = {
    "model": MODEL_NAME,
    "params_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
    "train_runtime_sec": round(train_result.metrics["train_runtime"], 1),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "val_accuracy": val_metrics["eval_Accuracy_score"],
    "val_macro_f1": val_metrics["eval_F1_score"],
    "test_accuracy": test_evaluation["eval_Accuracy_score"],
    "test_macro_f1": test_evaluation["eval_F1_score"],
}
print(json.dumps(results, indent=2))

Path("/content/outputs").mkdir(exist_ok=True)
with open("/content/outputs/distilbert_banking77_results.json", "w") as f:
    json.dump(results, f, indent=2)

{
  "model": "distilbert/distilbert-base-uncased",
  "params_millions": 67.0,
  "train_runtime_sec": 207.7,
  "best_checkpoint": "/content/outputs/distilbert-banking77/checkpoint-1255",
  "val_accuracy": 0.912543728135932,
  "val_macro_f1": 0.9156702066340762,
  "test_accuracy": 0.9175324675324675,
  "test_macro_f1": 0.9175972496791301
}


In [27]:
print(test_evaluation)

{'eval_loss': 0.3115241527557373, 'eval_Accuracy_score': 0.9175324675324675, 'eval_F1_score': 0.9175972496791301}


In [51]:
import torch
import torch.nn as nn

class ArtificialModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(10, 8)
        self.layer2 = nn.Linear(8, 4)
        self.output = nn.Linear(4, 2)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        return self.output(x)


model = ArtificialModel()

In [52]:
for name, param in model.named_parameters():
    print(name, param.shape, param.numel())

layer1.weight torch.Size([8, 10]) 80
layer1.bias torch.Size([8]) 8
layer2.weight torch.Size([4, 8]) 32
layer2.bias torch.Size([4]) 4
output.weight torch.Size([2, 4]) 8
output.bias torch.Size([2]) 2
